# NUST Bank Assistant – QLoRA Fine-Tuning with UnSloth

This notebook fine-tunes **Qwen2.5-3B-Instruct** on the NUST Bank product knowledge dataset using **UnSloth** (4-bit QLoRA).

**Requirements:** Run on Google Colab with a **T4 GPU** (free tier works).

## Steps
1. Install dependencies
2. Load & prepare training data from the bank dataset
3. Load Qwen2.5-3B-Instruct with UnSloth (4-bit quantization)
4. Fine-tune with QLoRA via SFTTrainer
5. Test inference
6. Save the LoRA adapter

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes xformers

## 2. Upload & Prepare Training Data

Upload `cleaned_chunks.json` (generated by `src/data_pipeline.py`) to Colab, then convert to instruction-tuning format.

In [ ]:
import json
from google.colab import files

# Upload cleaned_chunks.json
print("Upload cleaned_chunks.json:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

In [ ]:
# Convert chunks to instruction-tuning format
# For chunks that are Q&A pairs, extract question/answer
# For others, create a "summarize/explain" instruction

training_data = []

system_msg = (
    "You are a professional customer service assistant for NUST Bank. "
    "Answer questions about NUST Bank products and services accurately and politely. "
    "Only provide information from the bank's official documents. "
    "Never give financial advice."
)

for chunk in chunks:
    text = chunk["text"]
    meta = chunk.get("metadata", {})
    source = meta.get("sheet", meta.get("category", meta.get("source", "")))

    # Check if chunk is a Q&A pair
    if text.startswith("Q:") and "\nA:" in text:
        parts = text.split("\nA:", 1)
        question = parts[0].replace("Q:", "").strip()
        answer = parts[1].strip()
        if "\nNotes:" in answer:
            ans_parts = answer.split("\nNotes:", 1)
            answer = ans_parts[0].strip() + " " + ans_parts[1].strip()
    else:
        # Create a question about this information
        question = f"What can you tell me about {source}?" if source else "What information do you have about this bank product?"
        answer = text

    if len(question.strip()) < 5 or len(answer.strip()) < 5:
        continue

    training_data.append({
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    })

print(f"Created {len(training_data)} training examples")
print("\nSample:")
for msg in training_data[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}")

## 3. Load Model with UnSloth (4-bit Quantization)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters():,}")

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,          # optimized – 0 is faster
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves 30% VRAM
    random_state=42,
)

model.print_trainable_parameters()

## 4. Prepare Dataset for SFTTrainer

In [ ]:
from datasets import Dataset

# Convert to HF Dataset
def format_chat(example):
    """Apply the chat template to produce the final training text."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chat)

print(f"Dataset size: {len(dataset)}")
print(f"\nSample formatted text (first 500 chars):\n{dataset[0]['text'][:500]}")

## 5. Fine-Tune with QLoRA

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,  # pack short examples together for efficiency
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")

## 6. Test Inference

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

test_questions = [
    "What is the Little Champs Account?",
    "How can I open a Roshan Digital Account?",
    "What are the profit rates for savings accounts?",
    "How do I transfer funds using the mobile app?",
    "Tell me a joke",  # out-of-domain test
]

system_msg = (
    "You are a professional customer service assistant for NUST Bank. "
    "Answer questions about NUST Bank products and services accurately and politely. "
    "Only provide information from the bank's official documents. "
    "Never give financial advice."
)

for q in test_questions:
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    print(f"\nQ: {q}")
    print(f"A: {response}")
    print("-" * 60)

## 7. Save LoRA Adapter

In [ ]:
# Save locally
model.save_pretrained("nust_bank_lora_adapter")
tokenizer.save_pretrained("nust_bank_lora_adapter")
print("LoRA adapter saved to nust_bank_lora_adapter/")

# Download the adapter files
import shutil
shutil.make_archive("nust_bank_lora_adapter", "zip", "nust_bank_lora_adapter")
files.download("nust_bank_lora_adapter.zip")
print("Download started – save this adapter to use in the RAG pipeline.")

## 8. (Optional) Push to Hugging Face Hub

In [ ]:
# Uncomment and fill in your details to push to HF Hub
# model.push_to_hub("your-username/nust-bank-qwen2.5-3b-lora", token="hf_...")
# tokenizer.push_to_hub("your-username/nust-bank-qwen2.5-3b-lora", token="hf_...")